# 04c -- Stage 4c: Add AI-ArtBench (non-photographic/stylized AI-art gap)

Fine-tuned from the Stage 3b checkpoint, the custom model's training data
so far (DFDC face-swaps, CASIA2/Coverage/IMD2020 photographic forgeries,
DeepDetect-2025/DeepfakeArt AI content) is almost entirely photorealistic.
It has no exposure at all to stylized/illustrated/graphic-design AI
content -- which is exactly the gap that caused it to miss an AI-generated
**poster** (confirmed during manual testing, the finding that originally
motivated this whole Phase 1/2 expansion).

**[AI-ArtBench](https://www.kaggle.com/datasets/ravidussilva/real-ai-art)**
(`ravidussilva/real-ai-art` on Kaggle) directly fills that: 185,015 images
across 10 traditional art styles (Art Nouveau, Baroque, Expressionism,
Impressionism, Post-Impressionism, Realism, Renaissance, Romanticism,
Surrealism, Ukiyo-e), each style split three ways -- real human paintings,
Latent-Diffusion-generated, and Stable-Diffusion-generated -- so every
fake has a matching real counterpart in the exact same style, not just a
generic "AI art" bucket. License is reported as MIT on most sources, but
**double-check the license text on the live Kaggle listing yourself**
before relying on it -- one source (ieee-dataport.org) reported CC-BY
instead, and these two sources disagree.

**Honest gap this does NOT fully close** (documented in the project's own
expansion plan, not discovered here): there's no dataset specifically
labeled "AI poster vs. real poster" -- AI-ArtBench's painting styles are
adjacent to the poster problem (both are non-photographic/stylized AI
content a photorealism-trained detector has never seen), not identical to
it. This is the practical, Kaggle-sized step toward that gap, not a
complete fix.

**This notebook verifies the dataset's actual folder names before trusting
them** -- the exact literal casing/naming of AI-ArtBench's 30 per-split
subfolders couldn't be confirmed from outside Kaggle (only the 10-style x
3-source *structure* and total counts were cross-confirmed across
sources). Cell 5 below prints every discovered folder name so you can see
the real layout and the classifier's lenient keyword-matching logic is
confirmed against reality, rather than a guess, before anything gets used.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import os, random, zipfile
from pathlib import Path
import pandas as pd
from config.loader import get_settings, resolve_dataset_root, resolve_path

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Dataset root:", dataset_root)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def find_images(root):
    """os.walk(followlinks=True), not Path.glob -- same fix as Stage 2's
    DFDC extraction: pathlib's recurse_symlinks doesn't reliably follow
    Drive/Kaggle-mounted symlinks, plain os.walk does."""
    found = []
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        for fn in filenames:
            if Path(fn).suffix.lower() in IMAGE_EXTS:
                found.append(Path(dirpath) / fn)
    return found


def extract_zip_once(zip_name, dest_dir, marker_name=".extracted"):
    """Idempotent: skips re-extracting if dest_dir/marker_name already
    exists, so re-running this notebook doesn't redo a slow unzip."""
    dest_dir = Path(dest_dir)
    marker = dest_dir / marker_name
    if marker.exists():
        print(f"{dest_dir} already extracted, skipping.")
        return
    zip_path = None
    for candidate_root in (dataset_root, Path("."), Path("/content"), Path("/kaggle/input")):
        for p in Path(candidate_root).rglob(zip_name):
            zip_path = p
            break
        if zip_path:
            break
    if zip_path is None:
        print(f"Could not find {zip_name} -- upload it first. Skipping.")
        return
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path} -> {dest_dir} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)
    marker.touch()
    print("Done.")

## AI-ArtBench: locate the attached dataset

**Change `KAGGLE_INPUT_SLUG` below to match whatever "Add Input" shows
in your notebook's sidebar** -- same convention as Stage 2's DFDC
notebook. Kaggle slugifies a dataset's display title into its own
mount-folder name under `/kaggle/input/`, which does not necessarily
match the owner/dataset-name in its URL (`ravidussilva/real-ai-art`) --
a dataset shown as "AI-ArtBench" in the sidebar is more likely mounted
as `/kaggle/input/ai-artbench/` than `/kaggle/input/real-ai-art/`. The
next cell prints every folder actually under `/kaggle/input/` first, so
there's no need to guess -- read that output and adjust the slug below
if it doesn't match.

If you uploaded the zip yourself instead of using "Add Input", leave
`KAGGLE_INPUT_SLUG` as-is and unmatched -- this falls back to
`extract_zip_once("real-ai-art.zip", ...)`, which searches the dataset
root / current dir / `/content` / `/kaggle/input` for that filename.

In [ ]:
local_root = Path("/content/extra_datasets") if IS_COLAB else Path("/tmp/extra_datasets")
local_root.mkdir(parents=True, exist_ok=True)

if IS_KAGGLE and Path("/kaggle/input").exists():
    print("Folders currently under /kaggle/input/:")
    for p in sorted(Path("/kaggle/input").iterdir()):
        print(" ", p.name)

KAGGLE_INPUT_SLUG = "ai-artbench"  # <-- change to match the folder name printed above
kaggle_input_path = Path("/kaggle/input") / KAGGLE_INPUT_SLUG

if IS_KAGGLE and kaggle_input_path.exists():
    ai_artbench_dir = kaggle_input_path
    print("Using Kaggle-attached dataset directly:", ai_artbench_dir)
else:
    ai_artbench_dir = local_root / "ai_artbench"
    extract_zip_once("real-ai-art.zip", ai_artbench_dir, ".extracted")

## Verify the actual folder layout before trusting it

**Run this and read the output before continuing.** Prints every unique
directory name found anywhere under the extracted tree. Confirm each one
clearly signals either "real" or an AI source ("AI", "SD"/"stable",
"LD"/"latent") in its name -- if the real layout doesn't match that
assumption, the classifier two cells down needs adjusting before it's
trusted, not after.

In [ ]:
if ai_artbench_dir.exists():
    all_dirnames = sorted({p.name for p in ai_artbench_dir.rglob("*") if p.is_dir()})
    print(f"{len(all_dirnames)} unique directory names found:")
    for name in all_dirnames:
        print(" ", name)
else:
    print(f"{ai_artbench_dir} doesn't exist yet -- nothing to verify.")

## Classify real vs. AI-generated, with a sampling cap

**Structure confirmed from an actual run's printed output** (no longer a
guess): class folders are `AI_LD_<style>` / `AI_SD_<style>` (fake) and a
bare `<style>` name with no prefix at all (real) -- e.g. `baroque` is
real, `AI_SD_baroque` is fake. Everything else in the tree
(`Real_AI_SD_LD_Dataset`, `ravidussilva`, `real-ai-art`, `train`, `test`
-- artifacts of Kaggle's generic `datasets` mount path nesting owner/
dataset-name/top-level-folder several levels deep) is structural, not a
class folder, and must be ignored.

A first version matched by searching for "real"/"ai" substrings in the
immediate parent folder and one level up. That actually crashed on a
different bug (see the fixed cell above), but running it first surfaced
a second, subtler problem in its own right: Kaggle's generic mount path
inserts an extra `ravidussilva/real-ai-art/` nesting level before
`Real_AI_SD_LD_Dataset`, and that `real-ai-art` folder name happens to
contain "real" -- so bare-style-name (genuinely real) folders were only
getting classified correctly *by coincidence*, whenever a file happened
to sit the right number of levels under that one specific ancestor
name. Classifying by an explicit, confirmed style-name list instead of a
generic keyword search removes that coincidence entirely -- it no longer
matters how many extra nesting levels Kaggle's mount path happens to
insert, or what any of them happen to be named.

**`MAX_PER_CLASS` caps how many images get pulled in per real/fake class**
(randomly sampled, not just the first N encountered) -- AI-ArtBench's
185k images would otherwise dwarf every other dataset in the merged
manifest. Raise this later once there's room in the compute budget to
use more of it.

In [ ]:
MAX_PER_CLASS = 4000  # per real/fake class, not per style -- adjust as needed

# Confirmed from an actual Kaggle run's printed folder listing -- see the
# markdown above for why this replaced the earlier keyword-search version.
ART_STYLES = [
    "art_nouveau", "baroque", "expressionism", "impressionism",
    "post_impressionism", "realism", "renaissance", "romanticism",
    "surrealism", "ukiyo-e", "ukiyo_e",
]

def classify_real_fake(path):
    folder = path.parent.name.lower()
    if not any(style in folder for style in ART_STYLES):
        return None  # not a class folder at all (train/test/wrapper dirs/etc.)
    if folder.startswith("ai_ld_") or folder.startswith("ai_sd_") or folder.startswith("ai_"):
        return "fake"
    return "real"


ai_artbench_candidates = {"real": [], "fake": []}
unmatched = 0
if ai_artbench_dir.exists():
    for p in find_images(ai_artbench_dir):
        label = classify_real_fake(p)
        if label is None:
            unmatched += 1
            continue
        ai_artbench_candidates[label].append(p)

print(f"Matched: {len(ai_artbench_candidates['real'])} real, "
      f"{len(ai_artbench_candidates['fake'])} fake, {unmatched} unmatched")

random.seed(42)
ai_artbench_rows = []
for label, paths in ai_artbench_candidates.items():
    sampled = random.sample(paths, min(len(paths), MAX_PER_CLASS))
    manip_type = "none" if label == "real" else "gan"
    for p in sampled:
        ai_artbench_rows.append({"filename": str(p), "label": label, "type": manip_type})
    print(f"  sampled {len(sampled)}/{len(paths)} {label} images (cap={MAX_PER_CLASS})")

## Bundle the sampled images into one zip for Drive

Same reasoning as Stage 4b's DeepDetect-2025 cell: writing thousands of
individual small files straight onto Drive's FUSE mount is the slow/
fragile trap this project has already hit more than once. One zip is a
single fast write instead.

In [ ]:
if ai_artbench_rows:
    bundle_path = local_root / "ai_artbench_bundle.zip"
    bundle_rows = []
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for i, row in enumerate(ai_artbench_rows):
            src = Path(row["filename"])
            arcname = f"ai_artbench_{i:06d}{src.suffix}"
            zf.write(src, arcname)
            bundle_rows.append({**row, "filename": arcname})
        manifest_csv = pd.DataFrame(bundle_rows, columns=["filename", "label", "type"]).to_csv(index=False)
        zf.writestr("ai_artbench_manifest.csv", manifest_csv)
    print(f"Bundled {len(ai_artbench_rows)} images -> {bundle_path} "
          f"({bundle_path.stat().st_size / 1e6:.1f} MB)")
    if IS_COLAB:
        import shutil
        shutil.copy(bundle_path, dataset_root / bundle_path.name)
        print(f"Copied bundle to Drive: {dataset_root / bundle_path.name}")
else:
    print("No AI-ArtBench rows matched -- nothing to bundle (upload the zip and rerun from the extract cell).")

## Merge into the training manifest

**Check the printed counts above first.** Folds whatever AI-ArtBench rows
were matched into Stage 3's `merged_labels.csv` -- the same shared,
growing manifest every other stage builds onto, so a later retrain picks
up everything with no separate file to remember to point at. Re-running
this notebook replaces its previous rows instead of duplicating them.

**Rows keep their local-session path, same caveat as DeepDetect-2025's**:
only the bundle zip above is Drive-persisted, not these individual files
-- if you need these exact rows to survive into a future session, unzip
`ai_artbench_bundle.zip` from Drive and point the paths there first.

In [ ]:
merged_path = dataset_root / "merged_labels.csv"
base_df = pd.read_csv(merged_path) if merged_path.exists() else pd.DataFrame(columns=["filename", "label", "type"])

extra_df = pd.DataFrame(ai_artbench_rows, columns=["filename", "label", "type"])


def _is_ours(filename):
    return str(ai_artbench_dir) in filename


base_df = base_df[~base_df["filename"].astype(str).apply(_is_ours)]
merged_df = pd.concat([base_df, extra_df], ignore_index=True)
merged_df.to_csv(merged_path, index=False)
print(f"merged_labels.csv now has {len(merged_df)} rows ({len(extra_df)} from AI-ArtBench this run).")

## Next step

Rerun Stage 3's retrain cells (now reading the updated
`merged_labels.csv`) from the latest checkpoint -- same training loop
(augmentation on train only, sqrt-weighted oversampling, per-class +
balanced accuracy, best-balanced-accuracy checkpointing), no path to
change by hand. Re-test the specific AI-generated poster that originally
motivated this expansion once retrained, to confirm this actually closes
(or narrows) that gap rather than assuming it does.